# Stochastic Simulation — Exam Gap Analysis (2020–2025)

Based on inspection of all 6 past papers.

## 1. Topic Coverage vs. the High-Level Summary

### Well covered ✅ (appears in every recent exam)

| Topic | Location in summary |
|-------|---------------------|
| Inversion sampling (derive CDF, invert, pseudocode) | §2.2 |
| Rejection sampling (find $M$, optimal proposal, acceptance rate) | §2.3 |
| Transformation of RVs / Box–Muller | §2.4 |
| MC estimator (unbiasedness proof, variance, CLT) | §3.1 |
| IS estimator (variance derivation, optimal proposal minimisation) | §3.3 |
| MH algorithm — pseudocode + detailed balance proof | §4.3 |
| ULA/MALA — update step + limiting distribution derivation | §4.5 |
| Gibbs sampler | §4.4 |
| State-space models + Bootstrap Particle Filter pseudocode | §6 |
| Fisher's identity proof + gradient MMLE | §5.3 |

---

### Gaps ⚠️ (in notes but not in summary, or missing entirely)

| Topic | Years appeared | Risk |
|-------|---------------|------|
| Distribution of $N$ accepted from $K$ rejections: $N \sim \text{Binomial}(K, 1/M)$ | 2025 Q1(v) | **High** |
| Numerically stable $\log p(y_{1:t})$ in BPF (log-sum-exp trick) | 2025 Q4(b)(iv) | **High** |
| SIR smoother pseudocode (BPF storing full paths $x_{0:t}^{(i)}$) | 2025 Q4(b)(ii–iii) | **High** |
| Simulated annealing / SA as MCMC optimisation pseudocode | 2025 Q3(b), 2023 | **Medium** |
| Stochastic gradient / mini-batch ULA for large $n$ datasets | 2024 Q3(vi) | **Medium** |
| Self-normalised IS for marginal likelihood $p(y)$ — biased but consistent | 2025 Q2(a), 2023 Q2 | **Medium** |
| Antithetic variates | 2022 only | Low |
| ARS / ARMS / Slice sampling / Ratio of uniforms | 2020–2022 only | Very low (old lecturer era) |
| LCG period analysis | 2020–2022 only | Very low |

## 2. What You're Expected to *Do* (Derivation Mechanics)

The exam is **heavily derivation-based** — every question has at least one prove/derive sub-part. These are the recurring mechanics:

1. **Derive CDFs from scratch** (integration by parts, completing the square) then invert analytically.

2. **Find $M = \sup_x p(x)/q(x)$** by differentiating the ratio and solving — an optimal $\lambda$ or $\mu$ question appears in 3 of the last 4 years.

3. **Compute IS estimator variance in closed form** — requires a multi-step integral. The 2024 Q2 Rayleigh example is the canonical template:
$$\mathrm{Var}(\hat{\varphi}^N_{\mathrm{IS}}) = \frac{1}{N}\left[\int \varphi(x)^2 \frac{p(x)^2}{q(x)}\,dx - \bar{\varphi}^2\right].$$

4. **Prove Fisher's identity** — two-step proof:
$$\nabla_\theta \log p_\theta(y) = \frac{\nabla_\theta p_\theta(y)}{p_\theta(y)} = \frac{\int \nabla_\theta p_\theta(x,y)\,dx}{p_\theta(y)} = \int \nabla_\theta \log p_\theta(x,y)\cdot p_\theta(x|y)\,dx.$$

5. **Derive ULA stationary distribution** for a Gaussian target $p^\star = \mathcal{N}(\mu,\sigma^2)$: write the recursion as an AR(1) process, solve for the mean and variance in stationarity.

6. **Write BPF pseudocode correctly** — must include: propagate particles via $f$, compute weights $w^{(i)} \propto g(y_t|x_t^{(i)})$, normalise, resample.

7. **Prove MH detailed balance** — use the $\min(1,r)$ structure to show symmetry:
$$p^\star(x)\,\alpha(x,x')\,q(x'|x) = p^\star(x')\,\alpha(x',x)\,q(x|x').$$
Appears explicitly in 2023 Q3 and 2025 Q3.

## 3. The Three Gap Topics Most Likely to Appear Again

### Gap 1 — Distribution of $N$ accepted samples from $K$ iterations of rejection sampling

Each iteration accepts independently with probability $a = 1/M$. So:
$$N \sim \text{Binomial}(K,\, 1/M), \qquad \mathbb{E}[N] = K/M.$$

---

### Gap 2 — Numerically stable log-marginal likelihood in the BPF

The incremental marginal likelihood at step $t$ is estimated as:
$$\hat{p}(y_t|y_{1:t-1}) = \frac{1}{N}\sum_{i=1}^N g(y_t|x_t^{(i)}).$$

The full log-marginal likelihood accumulates:
$$\log p(y_{1:T}) = \sum_{t=1}^T \log \hat{p}(y_t|y_{1:t-1}).$$

**Numerical stability:** Compute each $\log \hat{p}(y_t|y_{1:t-1})$ using the **log-sum-exp** trick:
$$\log\frac{1}{N}\sum_i g(y_t|x_t^{(i)}) = \ell_{\max} + \log\frac{1}{N}\sum_i e^{\log g(y_t|x_t^{(i)}) - \ell_{\max}},$$
where $\ell_{\max} = \max_i \log g(y_t|x_t^{(i)})$. This prevents underflow when likelihoods are very small.

---

### Gap 3 — SIR Smoother pseudocode

The **Sequential Importance Resampling (SIR) smoother** is a BPF that stores full paths:

**Algorithm:**
1. At $t=0$: draw $x_0^{(i)} \sim \mu$, set $w_0^{(i)} = 1/N$.
2. For $t = 1,\ldots,T$:
   - Propagate: $x_t^{(i)} \sim f(x_t | x_{t-1}^{(i)})$ (or from proposal $q$).
   - Update unnormalised weight: $W_t^{(i)} = W_{t-1}^{(i)} \cdot \frac{f(x_t^{(i)}|x_{t-1}^{(i)})\,g(y_t|x_t^{(i)})}{q(x_t^{(i)}|x_{t-1}^{(i)})}$.
   - Normalise: $w_t^{(i)} = W_t^{(i)} / \sum_j W_t^{(j)}$.
   - **Resample** full paths $x_{0:t}^{(i)}$ with probabilities $w_t^{(i)}$, reset weights to $1/N$.
3. Return weighted paths $\{(x_{0:T}^{(i)}, w_T^{(i)})\}$.

**Use in gradient MMLE:** After running the smoother, approximate Fisher's identity:
$$\nabla_\theta \log p_\theta(y_{1:T}) \approx \sum_{i=1}^N w_T^{(i)} \sum_{t=1}^T \nabla_\theta \log p_\theta(x_{t-1}^{(i)}, x_t^{(i)}, y_t).$$

## 4. Marks Breakdown & 80% Assessment

**Format:** 4 questions × 20 marks = 80 marks total. Need **64/80 for 80%**.

| Question | Topic | Achievable marks | Notes |
|----------|-------|-----------------|-------|
| Q1 | Exact Sampling (inversion, rejection, transformation) | **17–19/20** | Very mechanical once practiced |
| Q2 | MC / IS (estimators, variance derivations, optimal proposals) | **15–18/20** | Algebra-heavy; 4–5 worked examples needed |
| Q3 | MCMC (MH proof, Gibbs, ULA/MALA, SA) | **14–17/20** | Proof of detailed balance is reproducible |
| Q4 | SSMs / Particle Filter (BPF, smoother, gradient) | **10–14/20** | Hardest; smoother + log-ML computation |

**Realistic total with focused prep: 56–68 / 80.**  80% is very achievable.

---

## 5. Recommended Practice Order

1. **Do 2025 Q1 + Q2 timed** — highest marks per hour, most repeatable structure.
2. **Memorise cold:** MH detailed balance proof + BPF pseudocode (write from memory until fluent).
3. **Work through 2024 Q2 fully** — the IS variance computation is the hardest algebra; mastering this structure covers 2023 Q2(c) and 2025 Q2(c) too.
4. **Add the three gap topics** to your notes (binomial $N$, log-sum-exp in BPF, SIR smoother).
5. **Attempt 2023 Q4 last** — state-space gradient + smoother in one question.